# Investment Portfolio Optimization using Python's SciPy Package

---

### Step 0: Setup

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

scipy minimize documentation: 
https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html

In [2]:
pd.options.display.max_columns = None

### Step 1: Prepare the Data

In [3]:
df = pd.read_excel("investment_portfolio_problem.xlsx", sheet_name="Data", usecols="A:C")
df = df.pivot(index='date', columns="stock", values="monthly return [%]")
display(df)

stock,AMZN,HD,WMT
date,,,
2014-07-01,8.322956,15.646248,2.609374
2014-08-01,-4.896179,-1.882359,1.952036
2014-09-01,-5.266099,6.837461,-0.261563
2014-10-01,10.862314,1.927804,14.776494
2014-11-01,-8.354007,5.603649,-1.896301
...,...,...,...
2019-03-01,8.185878,6.945029,6.011833
2019-04-01,-7.861329,-6.799203,-1.361337
2019-05-01,6.679177,9.544366,9.500147


In [4]:
mean=df.mean().tolist()
print(mean) 

[3.2301824721147545, 2.0356607382295073, 1.0806513842622953]


In [5]:
cov_matrix=df.cov(ddof=0)
cov_matrix

stock,AMZN,HD,WMT
stock,,,
AMZN,70.554009,18.399750,4.955124
HD,18.399750,27.680909,5.249985
WMT,4.955124,5.249985,28.979489


In [6]:
n = len(df.columns)

In [7]:
min_expected_return = 2.00 # for Model A
max_risk = 5.00 # for Model B

## Model A

### [A] Formulation

**Decision Variables:**

$x_{i}$ = fraction of investment allocated to stock $i=0,1, \ldots,n-1$

**Objective Function:**

Minimize $$ \sqrt{\sum_{i=0...,n-1} \sum_{j=0,...n-1} cov_{ij} \; x_{i} \; x_{j}} $$

**Constraints**

Subject to

\begin{align}
\text{(invest all)}&:& \sum_{i=0,...,n-1} x_{i} &= 1  \\
\text{(portfolio expected return)}&:& \sum_{i=0,...,n-1} \mu_{i} \; x_{i} &\geq 2.00 \\
\text{(nonnegativity)}&:& \; \; x_{i} &\geq 0, \; \; \; i=0,1, \ldots,n-1  
\end{align}



### [A] Step 2: Define the Objective Function

In [8]:
def risk_fn(x):
    risk_sum =0
    for i in range(n):
        for j in range(n):
            risk_sum +=x[i]*x[j]*cov_matrix.iloc[i,j]
    return np.sqrt(risk_sum)

### [A] Step 3: Define the Constraints (if any)

All *equality* constraint functions must be of the form: LHS = 0

For example:
$$ 
x_{0} + x_{1} + x_{2} = 1
$$

must be converted to:
$$ 
x_{0} + x_{1} + x_{2} -1 = 0
$$

In [9]:
#defining weights sum up to 1 equality constraint
def modA_eq_constraint_fn(x):
    return sum([x[i] for i in range(n)]) - 1

All *inequality* constraint functions must be of the form: LHS >= 0

In this example:
$$
 3.23 x_{0} + 2.04 x_{1} + 1.08 x_{2} - 2.00 \geq 0
$$

In [10]:
# defining minimum expected return inequality constraint
def modA_ineq_constraint_fn(x):
    return sum([mean[i] * x[i] for i in range(n)]) - min_expected_return

Then you need to **define the list of constraint dictionaries**:

In [11]:
modA_con1 = {"type": "eq", "fun": modA_eq_constraint_fn}
modA_con2 = {"type": "ineq", "fun": modA_ineq_constraint_fn}
modA_cons = ([modA_con1, modA_con2])

### [A] Step 4: Define the Bounds (if any)

In [12]:
#define bounds
b=(0,1)
modA_bnds=(b,)*n

In [13]:
modA_bnds

((0, 1), (0, 1), (0, 1))

In [14]:
## If there are no upper bounds:
# modA_bnds = ((0, None), (0, None), (0, None))

### [A] Step 5: Specify Initial Values

In [15]:
modA_x_initial = [1/n]*n

In [16]:
modA_x_initial

[0.3333333333333333, 0.3333333333333333, 0.3333333333333333]

### [A] Step 6: Solve

In [17]:
modA_sol = minimize(fun = risk_fn, 
                    x0 = modA_x_initial, 
                    method = "SLSQP", 
                    bounds = modA_bnds, 
                    constraints = modA_cons)

### [A] Step 7: Display / Print the Solution

In [18]:
print(modA_sol)
print("Objective Value (Risk):", modA_sol.fun)
print("Optimal Solution", modA_sol.x)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 4.277036848646487
           x: [ 2.232e-01  4.603e-01  3.165e-01]
         nit: 5
         jac: [ 6.029e+00  4.328e+00  2.968e+00]
        nfev: 21
        njev: 5
 multipliers: [ 1.429e+00  1.424e+00]
Objective Value (Risk): 4.277036848646487
Optimal Solution [0.22318256 0.4603209  0.31649654]


## Model B

### [B] Formulation

**Decision Variables:**

$x_{i}$ = fraction of investment allocated to stock $i=0,1, \ldots,n-1$

**Objective Function:**

Maximize $$ \sum_{i=0,...,n-1} \mu_{i} \; x_{i} $$

**Constraints**

Subject to

\begin{align}
\text{(invest all)}&:& \sum_{i=0,...,n-1} x_{i} &= 1  \\
\text{(portfolio risk)}&:& \sqrt{\sum_{i=0...,n-1} \sum_{j=0,...n-1} cov_{ij} \; x_{i} \; x_{j}} &\leq 5.00 \\
\text{(nonnegativity)}&:& \; \; x_{i} &\geq 0, \; \; \; i=0,1, \ldots,n-1  
\end{align}



### [B] Step 2: Define the Objective Function

In [19]:
def expected_return_fn(x):
    expected_return_sum =0
    for i in range(n):
        expected_return_sum += mean[i] * x[i]
    return -expected_return_sum #negate to convert to minimization problem

### [B] Step 3: Define the Constraints (if any)

All *equality* constraint functions must be of the form: LHS = 0

For example:
$$ 
x_{0} + x_{1} + x_{2} = 1
$$

must be converted to:
$$ 
x_{0} + x_{1} + x_{2} -1 = 0
$$

In [20]:
#defining weights sum up to 1 equality constraint
def modB_eq_constraint_fn(x):
    return sum([x[i] for i in range(n)]) - 1

All *inequality* constraint functions must be of the form: LHS >= 0

In this example:
$$
5.00 - \sqrt{\sum_{i=0...,n-1} \sum_{j=0,...n-1} cov_{ij} \; x_{i} \; x_{j}} \geq 0
$$

In [21]:
# defining minimum expected return inequality constraint
def modB_ineq_constraint_fn(x):
    return max_risk - risk_fn(x)

Then you need to **define the list of constraint dictionaries**:

In [22]:
modB_con1 = {"type": "eq", "fun": modB_eq_constraint_fn}
modB_con2 = {"type": "ineq", "fun": modB_ineq_constraint_fn}
modB_cons = ([modB_con1, modB_con2])

### [B] Step 4: Define the Bounds (if any)

In [23]:
#define bounds
b=(0,1)
modB_bnds=(b,)*n

In [24]:
## If there are no upper bounds:
# modB_bnds = ((0, None), (0, None), (0, None))

### [B] Step 5: Specify Initial Values

In [25]:
modB_x_initial = [1/n]*n

### Step 6: Solve

In [26]:
modB_sol = minimize(fun=expected_return_fn, 
                    x0=modB_x_initial, 
                    method="SLSQP", 
                    bounds=modB_bnds, 
                    constraints=modB_cons)

### Step 7: Display / Print the Solution

In [27]:
print(modB_sol)
print("Objective Value (Expected Return):", -modB_sol.fun)
print("Optimal Solution", modB_sol.x)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: -2.3528727686494673
           x: [ 3.811e-01  4.744e-01  1.445e-01]
         nit: 6
         jac: [-3.230e+00 -2.036e+00 -1.081e+00]
        nfev: 24
        njev: 6
 multipliers: [-4.174e-01  3.871e-01]
Objective Value (Expected Return): 2.3528727686494673
Optimal Solution [0.38108822 0.47440415 0.14450763]
